In [0]:
%run ./00_common

In [0]:
import statistics
import time
from pyspark.sql import functions as F
import pandas as pd


Common loaded


In [0]:
# Fair timing: disable local disk caching to force storage reads
spark.conf.set("spark.databricks.io.cache.enabled", "false")

# Load base fact table
base = spark.read.format("delta").load(gold_path("fact_order_items"))
print("Base rows count:", base.count())

# Multiply rows by 50 using crossJoin on a range
big = (
    base.crossJoin(spark.range(50).withColumnRenamed("id", "copy_id"))
        .withColumn("purchase_month", (F.col("purchase_date_key") / 100).cast("int"))
)

# Define test storage paths inside gold/_perf
before_p = path("gold", "_perf", "items_before")
after_p = path("gold", "_perf", "items_after")

# BEFORE: 400 small files with mixed, unclustered data (no partitioning)
big.repartition(400).write.format("delta").mode("overwrite").save(before_p)
print("BEFORE table created successfully at:", before_p)

Base rows count: 98309
BEFORE table created successfully at: abfss://gold@stretaillakedsp01.dfs.core.windows.net/_perf/items_before


In [0]:
def detail(p):
    """Returns total file count and size in MB for a Delta table path."""
    r = spark.sql(f"DESCRIBE DETAIL delta.`{p}`").first()
    return r["numFiles"], round(r["sizeInBytes"] / 1024 / 1024, 1)

# Find the highest volume product_key to query against
top = base.groupBy("product_key").count().orderBy(F.desc("count")).first()["product_key"]

def queries(p):
    """Returns standard benchmark queries targeting the path."""
    t = spark.read.format("delta").load(p)
    return {
        "Q1 one product": t.filter(F.col("product_key") == top).agg(F.sum("price")),
        "Q2 one month": t.filter(F.col("purchase_month") == 201801).agg(F.sum("price")),
        "Q3 one product + month": t.filter((F.col("product_key") == top) & (F.col("purchase_month") == 201801)).agg(F.sum("price")),
        "Q4 top 10 sellers": t.groupBy("seller_key").agg(F.sum("price").alias("revenue")).orderBy(F.desc("revenue")).limit(10),
    }

def run(p, runs=3):
    """Runs each query multiple times using noop format write to trigger execution, returning median seconds."""
    out = {}
    for name, df in queries(p).items():
        times = []
        for _ in range(runs):
            t0 = time.time()
            df.write.format("noop").mode("overwrite").save()  # Executes query plan without disk write overhead
            times.append(time.time() - t0)
        out[name] = round(statistics.median(times), 2)
    return out

# Execute baseline measurement
before_res, before_detail = run(before_p), detail(before_p)
print("BEFORE (Files, Total MB):", before_detail)
print("BEFORE Query Durations (sec):", before_res)

BEFORE (Files, Total MB): (400, 334.8)
BEFORE Query Durations (sec): {'Q1 one product': 3.95, 'Q2 one month': 3.68, 'Q3 one product + month': 3.29, 'Q4 top 10 sellers': 4.35}


In [0]:
(big.write.format("delta").mode("overwrite")
    .partitionBy("purchase_month")
    .save(after_p))

spark.sql(f"OPTIMIZE delta.`{after_p}` ZORDER BY (product_key)")

after_res, after_detail = run(after_p), detail(after_p)
print("AFTER (Files, Total MB):", after_detail)
print("AFTER Query Durations (sec):", after_res)

AFTER (Files, Total MB): (21, 7.8)
AFTER Query Durations (sec): {'Q1 one product': 0.61, 'Q2 one month': 0.47, 'Q3 one product + month': 0.47, 'Q4 top 10 sellers': 1.86}


In [0]:

cmp = pd.DataFrame({"before_sec": before_res, "after_sec": after_res})
cmp["faster_x"] = (cmp["before_sec"] / cmp["after_sec"]).round(1)

print("Files and Size (MB) -> BEFORE:", before_detail, " | AFTER:", after_detail)
display(cmp)

Files and Size (MB) -> BEFORE: (400, 334.8)  | AFTER: (21, 7.8)


before_sec,after_sec,faster_x
3.95,0.61,6.5
3.68,0.47,7.8
3.29,0.47,7.0
4.35,1.86,2.3


In [0]:
# Compact Bronze raw file fragments
bp = bronze_path("order_items")
print("Bronze order_items BEFORE (Files, MB):", detail(bp))

spark.sql(f"OPTIMIZE delta.`{bp}`")
print("Bronze order_items AFTER (Files, MB):", detail(bp))

# Enable Auto-Optimize table properties on production Gold fact tables
for t in ["fact_orders", "fact_order_items"]:
    spark.sql(f"""ALTER TABLE delta.`{gold_path(t)}` SET TBLPROPERTIES (
        delta.autoOptimize.optimizeWrite = true,
        delta.autoOptimize.autoCompact = true)""")
print("Auto-Optimize properties successfully set on Gold fact tables!")

Bronze order_items BEFORE (Files, MB): (31, 5.6)
Bronze order_items AFTER (Files, MB): (31, 5.6)
Auto-Optimize properties successfully set on Gold fact tables!
